# Dalla softmax alla ricorrenza

Il codice del capitolo [«Dalla softmax alla ricorrenza»](https://book.paithon.it/main/AttenzioneLineare/dalla-softmax-alla-ricorrenza.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Dalla softmax alla ricorrenza

[Leggi la pagina](https://book.paithon.it/main/AttenzioneLineare/dalla-softmax-alla-ricorrenza.html)


### Addestrare in parallelo, generare in ricorrenza


In [ ]:
import torch
import torch.nn.functional as F

phi = lambda x: F.elu(x) + 1.0   # feature map: sempre positiva

def genera_passo(S, z, q_t, k_t, v_t):
    """Un passo di attenzione lineare, forma ricorrente.
    S: stato d x d (fisso)   z: normalizzatore d   q_t, k_t, v_t: vettori d."""
    pk = phi(k_t)
    S = S + torch.outer(v_t, pk)          # scrivi: S += v_t phi(k_t)^T
    z = z + pk                            # aggiorna il normalizzatore
    pq = phi(q_t)
    o_t = (S @ pq) / (z @ pq)             # leggi: o_t = S phi(q_t) / (z . phi(q_t))
    return o_t, S, z

### Il limite dell'accumulo


In [ ]:
import numpy as np

def errore_richiamo(N, d=32, prove=2000, seme=0):
    # errore relativo medio nel rileggere N associazioni da una memoria d x d
    rng = np.random.default_rng(seme)
    errori = []
    for _ in range(prove):
        K = rng.standard_normal((N, d))
        K /= np.linalg.norm(K, axis=1, keepdims=True)  # chiavi a norma unitaria
        V = rng.standard_normal((N, d))
        S = V.T @ K                                    # la memoria dopo N scritture
        letto = (S @ K.T).T                            # rileggo con le stesse chiavi
        errori.append((np.linalg.norm(letto - V, axis=1)
                       / np.linalg.norm(V, axis=1)).mean())
    return float(np.mean(errori))

for N in (8, 16, 32):
    print(f"N={N:2d}  errore relativo medio = {errore_richiamo(N):.2f}"
          f"   (andamento atteso ~{np.sqrt((N - 1) / 32):.2f})")

## Scrivere meglio nella memoria: gate e delta rule

[Leggi la pagina](https://book.paithon.it/main/AttenzioneLineare/scrivere-nella-memoria.html)


### Correggere: la delta rule


In [ ]:
import numpy as np

def scrivi(K, V, regola):
    """Memoria d x d dopo aver scritto le coppie (k, v) in ordine."""
    S = np.zeros((K.shape[1], K.shape[1]))
    for k, v in zip(K, V):
        scritto = v - S @ k if regola == "delta" else v   # delta: l'errore
        S += np.outer(scritto, k)
    return S

def prova(N, d=32, prove=2000, seme=0):
    rng = np.random.default_rng(seme)
    err = {"somma": [], "delta": []}     # richiamo delle N associazioni
    nuovo = {"somma": [], "delta": []}   # valore riscritto sotto la chiave 0
    for _ in range(prove):
        K = rng.standard_normal((N, d))
        K /= np.linalg.norm(K, axis=1, keepdims=True)   # chiavi di norma 1
        V = rng.standard_normal((N, d))
        v_nuovo = rng.standard_normal(d)
        for regola in err:
            S = scrivi(K, V, regola)
            letto = (S @ K.T).T
            err[regola].append((np.linalg.norm(letto - V, axis=1)
                                / np.linalg.norm(V, axis=1)).mean())
            k0 = K[0]
            scritto = v_nuovo - S @ k0 if regola == "delta" else v_nuovo
            S += np.outer(scritto, k0)
            nuovo[regola].append(np.linalg.norm(S @ k0 - v_nuovo)
                                 / np.linalg.norm(v_nuovo))
    return {r: (np.mean(err[r]), np.mean(nuovo[r])) for r in err}

for N in (8, 16, 32):
    r = prova(N)
    print(f"N={N:2d}  richiamo: somma {r['somma'][0]:.2f}, "
          f"delta {r['delta'][0]:.2f}  | riscrittura: "
          f"somma {r['somma'][1]:.2f}, delta {r['delta'][1]:.2f}")